# iRage AlgoArena — Short-Horizon Return Prediction | v6 FINAL

ONE SUBMISSION ONLY — ALL 6 ROOT-CAUSE FIXES APPLIED

## v6 Changes vs v5 (Evidence-Based from v5 Logs)

| Fix | v5 | v6 | Evidence |
|---|---|---|---|
| min_gain_to_split | 0.01 | **0.0** | LGB stopped at round 22 — threshold pruned real signal |
| min_child_samples | 300 | **100** | Too coarse for pattern detection in 661k rows |
| lambda_l2 | 10.0 | **3.0** | Killed tiny-but-real gradients in R2~0.001 regime |
| LGB LR / rounds | 0.05 / 2500 | **0.02 / 5000** | Model needs more rounds now params are fixed |
| Target clip | P1/P99 | **P2/P98** | P1/P99 added noise rows [-0.097, 0.097] |
| Feature selection | None (916) | **Top 99% gain** | 22 splits used across 916 features = ~890 wasted |
| Z-score standardize | No | **Train-stats only** | Inter-family scale mismatch hurts gradient comparability |
| CAT min_data_in_leaf | 300 | **100** | Same reasoning as LGB |
| CAT l2_leaf_reg | 15.0 | **5.0** | GroupKFold is honest; over-regularizing hurt signal |

## Runtime Budget (30-min = 1800s)

| Phase | Est. Time |
|---|---|
| Data load + FE | ~60s |
| Feature selection (1 fold, 500 rounds) | ~30s |
| Z-score standardization | ~5s |
| LGB: 5 folds x ~500 rounds (LR=0.02, early_stop=200) | ~300s |
| CatBoost: 5 folds x 1200 iter (LR=0.04, od_wait=100) | ~1385s |
| Ensemble + save | ~20s |
| Total | **~1800s (27-29 min)** |

## Evaluation Formula
Final Score = 2 x mean_batch_R2 - full_test_R2
Strategy: consistent batch generalization, zero structural leakage.

In [1]:
# CELL 1 - IMPORTS
import os, gc, warnings, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
from scipy.optimize import minimize

import lightgbm as lgb
from sklearn.model_selection import GroupKFold, KFold
from sklearn.metrics import r2_score

try:
    from catboost import CatBoostRegressor, Pool
    CATBOOST_AVAILABLE = True
    print('CatBoost OK')
except ImportError:
    CATBOOST_AVAILABLE = False
    print('CatBoost not installed - LGB only')

SEED = 23
np.random.seed(SEED)
NOTEBOOK_START = time.time()

print(f'LightGBM {lgb.__version__}')
print('Imports OK')

CatBoost OK
LightGBM 4.6.0
Imports OK


In [2]:
# CELL 2 - CONFIG (v6)
#
# v6 CHANGES vs v5 (all evidence-based from v5 logs):
#
# FIX 1: min_gain_to_split 0.01 -> 0.0  [CRITICAL]
#   v5 LGB stopped at round 22. Root cause: min_gain_to_split=0.01 refused
#   splits gaining <0.01, but in a R2~0.001 problem valid splits gain ~0.001.
#   Setting to 0.0 allows all statistically valid splits. Early stopping
#   prevents overfitting; we do NOT need a gain threshold on top.
#
# FIX 2: min_child_samples 300 -> 100  [CRITICAL]
#   300 samples/leaf = too coarse. Each leaf needs 9k+ rows from 661k.
#   Reduces to 100: still large enough to prevent overfitting, fine enough
#   to detect real patterns in subpopulations.
#
# FIX 3: lambda_l2 10.0 -> 3.0  [HIGH]
#   At true signal R2~0.001, lambda_l2=10 kills the tiny-but-real gradient
#   signals. Reduce to 3: still regularizing, not signal-killing.
#
# FIX 4: LR 0.05 -> 0.02, rounds 2500 -> 5000, early_stop 100 -> 200  [HIGH]
#   With min_gain_to_split fixed, model will now run more rounds.
#   LR=0.02 with early_stop=200: enough patience to find real signal.
#   5000 cap: model will stop ~300-800 now (not 22).
#   Time estimate: ~200s/fold x 5 = ~17 min (safe within 30 min budget).
#
# FIX 5: TARGET_CLIP P1/P99 -> P2/P98  [MEDIUM]
#   v5 log: P1/P99 clip = [-0.097, 0.097]. v4 P2/P98 = [-0.076, 0.076].
#   The 1-2% range is extreme microstructure noise (kurtosis=48).
#   P2/P98 is the proven tighter clip. Revert.
#
# FIX 6: Feature selection via OOF importance (new cell inserted after FE)
#   Select top features by cumulative gain importance >= 99% threshold.
#   Reduces from 916 to ~150-300 features.
#   Eliminates: collinear momentum combos, zero-importance noise.
#
# FIX 7: Z-score standardization using TRAIN-ONLY stats  [MEDIUM]
#   Per-feature: mean/std from training data applied to both train and test.
#   NOT a policy violation: uses only training statistics (no cross-sample
#   test operations). Removes inter-family scale differences so gradient
#   magnitudes are comparable across features.
#
# CatBoost: params unchanged, still PRIMARY model.
# v4 CAT OOF 0.055 was with KFold (leaked). GroupKFold will give true estimate.

class CFG:
    if os.path.exists('/kaggle/input'):
        DATA_DIR = Path('/kaggle/input/competitions/short-horizon-return-prediction-challenge-by-i-rage')
    else:
        DATA_DIR = Path(r'F:\#Project\iRAGE_Hackathon\algo5\kaggle\input\competitions\short-horizon-return-prediction-challenge-by-i-rage')
    TRAIN_PATH = DATA_DIR / 'train.parquet'
    TEST_PATH  = DATA_DIR / 'test.parquet'
    SAMPLE_SUB = DATA_DIR / 'sample_submission.csv'
    if os.path.exists('/kaggle/working'):
        OUTPUT_DIR = Path('/kaggle/working')
    else:
        OUTPUT_DIR = Path('./output')
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    N_FOLDS = 5
    SEED    = 23

    # FIX 5: Reverted to P2/P98. P1/P99 added noise rows not signal.
    TARGET_CLIP = True
    CLIP_LOWER  = 2
    CLIP_UPPER  = 98

    BUILD_MOMENTUM     = True
    BUILD_ABS_MOMENTUM = True
    BUILD_FAMILY_AGG   = True
    BUILD_PRICE_FEATS  = True
    BUILD_SO3_POLY     = True

    # FIX 6: Feature selection threshold
    # Keep features covering top FEAT_IMPORTANCE_THRESH cumulative gain.
    FEAT_IMPORTANCE_THRESH = 0.99   # top 99% of total gain
    FEAT_MIN_KEEP          = 50     # always keep at least 50 features

    # FIX 7: Z-score standardization using train-only stats
    STANDARDIZE = True

    # LGB - FIXED PARAMS (v6)
    # CRITICAL FIXES: min_gain_to_split=0.0, min_child_samples=100, lambda_l2=3
    # These 3 params caused LGB to stop at round 22 in v5 (100x below true signal).
    LGB_PARAMS = {
        'objective':         'regression',
        'metric':            'rmse',
        'learning_rate':     0.02,      # FIX 4: slower LR, more patience
        'num_leaves':        31,
        'max_depth':         -1,
        'min_child_samples': 100,       # FIX 2: was 300, too coarse
        'feature_fraction':  0.6,
        'bagging_fraction':  0.8,
        'bagging_freq':      1,
        'lambda_l1':         0.5,
        'lambda_l2':         3.0,       # FIX 3: was 10.0, killed tiny signals
        'min_gain_to_split': 0.0,       # FIX 1: was 0.01, pruned valid splits
        'verbose':           -1,
        'n_jobs':            -1,
        'seed':              SEED,
    }
    LGB_NUM_ROUNDS = 5000           # FIX 4: model will now actually use rounds
    LGB_EARLY_STOP = 200            # FIX 4: more patience

    # CatBoost - PRIMARY MODEL (params unchanged from v5)
    # min_data_in_leaf=100 (was 300) - same reasoning as LGB fix
    # l2_leaf_reg=5 (was 15) - less aggressive given GroupKFold CV
    CAT_PARAMS = dict(
        iterations=1200,
        learning_rate=0.04,
        depth=5,
        l2_leaf_reg=5.0,            # relaxed from 15: GroupKFold is honest
        min_data_in_leaf=100,       # FIX: was 300, too coarse
        border_count=128,
        random_seed=SEED,
        eval_metric='RMSE',
        od_type='Iter',
        od_wait=100,
        verbose=200,
        task_type='CPU',
        thread_count=-1,
        allow_writing_files=False,
    )

    RUNTIME_LIMIT_S   = 1620
    CAT_SECS_PER_ITER = 0.231

print('Config OK')

Config OK


In [3]:
# CELL 3 - DATA LOADING
print('Loading data...')
t0 = time.time()

train = pd.read_parquet(CFG.TRAIN_PATH)
test  = pd.read_parquet(CFG.TEST_PATH)

print(f'Train: {train.shape} | Test: {test.shape}')
print(f'Loaded in {time.time()-t0:.1f}s')

train_ids = train['ID'].values.copy()
test_ids  = test['ID'].values.copy()
y_raw     = train['TARGET'].copy()

# CV_GROUP: each group = independent market regime (not guaranteed chronological).
# GroupKFold on CV_GROUP matches the competition's randomized-batch evaluation.
# Host confirmed: treat as independent regimes.
if 'CV_GROUP' in train.columns:
    cv_groups    = train['CV_GROUP'].values.copy()
    USE_GROUP_CV = True
    n_groups     = len(np.unique(cv_groups))
    print(f'CV_GROUP found: {n_groups} groups -> GroupKFold')
    train.drop(columns=['ID', 'TARGET', 'CV_GROUP'], inplace=True)
else:
    cv_groups    = None
    USE_GROUP_CV = False
    print('CV_GROUP not found -> KFold fallback')
    train.drop(columns=['ID', 'TARGET'], inplace=True)

test.drop(columns=['ID'], inplace=True)

assert list(train.columns) == list(test.columns), \
    f'FATAL: train/test column mismatch! {len(train.columns)} vs {len(test.columns)}'

print(f'Features: {train.shape[1]}')
print(f'NaN train: {train.isnull().sum().sum():,} | NaN test: {test.isnull().sum().sum():,}')
print(f'Train IDs: {train_ids.min()} to {train_ids.max()}')
print(f'Test IDs:  {test_ids.min()} to {test_ids.max()}')
print(f'Elapsed: {time.time()-NOTEBOOK_START:.0f}s')

Loading data...
Train: (661574, 448) | Test: (410139, 446)
Loaded in 13.8s
CV_GROUP found: 70 groups -> GroupKFold
Features: 445
NaN train: 0 | NaN test: 0
Train IDs: 0 to 672373
Test IDs:  672374 to 1089244
Elapsed: 24s


In [4]:
# CELL 4 - TARGET ANALYSIS AND CLIPPING
# TARGET = 100 * (Price[t+H] - Price[t]) / Price[t]
# Fat-tailed (kurtosis >> 3). P1/P99 clip retains more tail signal than P2/P98.
# TRAINING-ONLY clip. Test predictions fully unconstrained.

desc = y_raw.describe(percentiles=[0.01, 0.02, 0.05, 0.25, 0.5, 0.75, 0.95, 0.98, 0.99])
print('TARGET DISTRIBUTION (raw)')
print(desc)
print(f'Skewness: {y_raw.skew():.4f} | Kurtosis: {y_raw.kurtosis():.4f}')

if CFG.TARGET_CLIP:
    clip_lo   = y_raw.quantile(CFG.CLIP_LOWER / 100)
    clip_hi   = y_raw.quantile(CFG.CLIP_UPPER / 100)
    y_series  = y_raw.clip(clip_lo, clip_hi)
    n_clipped = int(((y_raw < clip_lo) | (y_raw > clip_hi)).sum())
    print(f'Clip: [{clip_lo:.6f}, {clip_hi:.6f}]  ({n_clipped:,} rows = {100*n_clipped/len(y_raw):.2f}%)')
    print(f'Std: {y_raw.std():.6f} -> {y_series.std():.6f}')
    y = y_series.values.astype(np.float32)
else:
    y = y_raw.values.astype(np.float32)

print('Target ready OK')

TARGET DISTRIBUTION (raw)
count    661574.000000
mean         -0.000036
std           0.036660
min          -1.285424
1%           -0.097335
2%           -0.075900
5%           -0.051777
25%          -0.016153
50%           0.000000
75%           0.016073
95%           0.051537
98%           0.075723
99%           0.097421
max           1.346982
Name: TARGET, dtype: float64
Skewness: 0.1691 | Kurtosis: 48.0903
Clip: [-0.075900, 0.075723]  (26,464 rows = 4.00%)
Std: 0.036660 -> 0.029513
Target ready OK


In [5]:
# CELL 5 - FEATURE CATEGORIZATION
all_cols  = train.columns.tolist()

lag1_cols = [c for c in all_cols if c.endswith('_LagT1')]
lag2_cols = [c for c in all_cols if c.endswith('_LagT2')]
lag3_cols = [c for c in all_cols if c.endswith('_LagT3')]
lag_set   = set(lag1_cols + lag2_cols + lag3_cols)
base_cols = [c for c in all_cols if c not in lag_set]

# SO3_T = published covariate (session time marker)
# Price = target driver (TARGET = f(Price future))
special_cols   = ['Price', 'SO3_T']
pure_base_cols = [c for c in base_cols if c not in special_cols]

def get_family(col):
    return col.split('_')[0]

families = sorted(set(get_family(c) for c in pure_base_cols))

lag_map = {
    c: {
        'lag1': c + '_LagT1' if (c + '_LagT1') in lag_set else None,
        'lag2': c + '_LagT2' if (c + '_LagT2') in lag_set else None,
        'lag3': c + '_LagT3' if (c + '_LagT3') in lag_set else None,
    }
    for c in pure_base_cols
}

print(f'Base incl special: {len(base_cols):4d}')
print(f'Pure base:         {len(pure_base_cols):4d}')
print(f'LagT1/T2/T3:       {len(lag1_cols)}/{len(lag2_cols)}/{len(lag3_cols)}')
print(f'Families:          {families}')
print(f'Total orig feats:  {len(all_cols):4d}')
print('Categorization OK')

Base incl special:  112
Pure base:          110
LagT1/T2/T3:       111/111/111
Families:          ['S01', 'S02', 'S03', 'S04', 'S05']
Total orig feats:   445
Categorization OK


In [6]:
# CELL 6 - FEATURE ENGINEERING
#
# ALL feature engineering is PURELY ROW-WISE.
# No cross-sample operations. Same function applied to train and test identically.
# No test-set ranking, sorting, clustering, or neighborhood operations.
#
# QUANT RATIONALE:
# Acc12 = LagT1 - LagT2: short-term momentum acceleration
# Acc23 = LagT2 - LagT3: medium-term momentum acceleration
# Div13 = LagT1 - LagT3: full-span trend divergence
#   Accelerating momentum -> continuation signal
#   Decelerating -> reversal warning
#
# |LagT1| (AbsMom): momentum magnitude
#   High magnitude = strong regime -> stronger continuation
#   Tree models cannot learn |x| from x alone without this feature
#
# Family BaseMean/Std, MomMean/Std/Spread:
#   Mean: cross-instrument consensus level/direction
#   Std: disagreement within family = regime stress / divergence
#   Spread(LagT1): max-min of momentum = coordination vs divergence signal
#
# Price features: same logic on the TARGET-driving variable
#
# SO3_T polynomial (sq, cubic):
#   Session-time marker with nonlinear intraday return patterns
#   Quadratic: U-shape (high at open/close)
#   Cubic: asymmetric morning vs afternoon drift
#
# NOT BUILT:
#   Ratios (X/Y): anonymous features have unknown scale -> division instability
#   Pairwise products: 800^2 = 640k features -> dimensionality curse
#   Any test-set cross-sample statistics

def feature_engineering(df, lag_map, families, pure_base_cols, verbose=True):
    df = df.copy()
    new_feats = []

    # A. MOMENTUM
    if CFG.BUILD_MOMENTUM:
        n0 = len(new_feats)
        for base_col, lags in lag_map.items():
            l1, l2, l3 = lags['lag1'], lags['lag2'], lags['lag3']
            if l1 and l2 and l1 in df.columns and l2 in df.columns:
                df[base_col + '_Acc12'] = df[l1] - df[l2]
                new_feats.append(base_col + '_Acc12')
            if l2 and l3 and l2 in df.columns and l3 in df.columns:
                df[base_col + '_Acc23'] = df[l2] - df[l3]
                new_feats.append(base_col + '_Acc23')
            if l1 and l3 and l1 in df.columns and l3 in df.columns:
                df[base_col + '_Div13'] = df[l1] - df[l3]
                new_feats.append(base_col + '_Div13')
        if verbose: print(f'  Momentum (Acc12/Acc23/Div13): {len(new_feats)-n0}')

    # B. ABSOLUTE MOMENTUM
    if CFG.BUILD_ABS_MOMENTUM:
        n0 = len(new_feats)
        for base_col, lags in lag_map.items():
            l1 = lags['lag1']
            if l1 and l1 in df.columns:
                df[base_col + '_AbsMom'] = df[l1].abs()
                new_feats.append(base_col + '_AbsMom')
        if verbose: print(f'  AbsMom (|LagT1|):             {len(new_feats)-n0}')

    # C. PRICE-SPECIFIC
    if CFG.BUILD_PRICE_FEATS and 'Price' in df.columns:
        n0 = len(new_feats)
        p_l1, p_l2, p_l3 = 'Price_LagT1', 'Price_LagT2', 'Price_LagT3'
        if p_l1 in df.columns and p_l2 in df.columns:
            df['Price_Acc']    = df[p_l1] - df[p_l2]; new_feats.append('Price_Acc')
        if p_l1 in df.columns and p_l3 in df.columns:
            df['Price_Div13']  = df[p_l1] - df[p_l3]; new_feats.append('Price_Div13')
        if p_l2 in df.columns and p_l3 in df.columns:
            df['Price_Acc23']  = df[p_l2] - df[p_l3]; new_feats.append('Price_Acc23')
        if p_l1 in df.columns:
            df['Price_AbsMom'] = df[p_l1].abs();       new_feats.append('Price_AbsMom')
        if verbose: print(f'  Price-specific:               {len(new_feats)-n0}')

    # D. FAMILY AGGREGATES
    if CFG.BUILD_FAMILY_AGG:
        n0 = len(new_feats)
        for fam in families:
            fam_base = [c for c in pure_base_cols if get_family(c) == fam and c in df.columns]
            fam_lag1 = [c + '_LagT1' for c in fam_base if c + '_LagT1' in df.columns]
            if len(fam_base) < 2: continue
            df[fam + '_BaseMean'] = df[fam_base].mean(axis=1)
            df[fam + '_BaseStd']  = df[fam_base].std(axis=1)
            new_feats += [fam + '_BaseMean', fam + '_BaseStd']
            if len(fam_lag1) >= 2:
                fam_mom = df[fam_lag1]
                df[fam + '_MomMean']   = fam_mom.mean(axis=1)
                df[fam + '_MomStd']    = fam_mom.std(axis=1)
                df[fam + '_MomSpread'] = fam_mom.max(axis=1) - fam_mom.min(axis=1)
                new_feats += [fam + '_MomMean', fam + '_MomStd', fam + '_MomSpread']
        if verbose: print(f'  Family aggregates:            {len(new_feats)-n0}')

    # E. SO3_T POLYNOMIAL + INTERACTIONS
    if CFG.BUILD_SO3_POLY and 'SO3_T' in df.columns:
        n0 = len(new_feats)
        df['SO3_T_sq'] = df['SO3_T'] ** 2;  new_feats.append('SO3_T_sq')
        df['SO3_T_cb'] = df['SO3_T'] ** 3;  new_feats.append('SO3_T_cb')
        if 'Price_LagT1' in df.columns:
            df['SO3T_x_PriceMom'] = df['SO3_T'] * df['Price_LagT1']
            new_feats.append('SO3T_x_PriceMom')
        for fam in families:
            fname = fam + '_MomMean'
            if fname in df.columns:
                df['SO3T_x_' + fam + '_Mom'] = df['SO3_T'] * df[fname]
                new_feats.append('SO3T_x_' + fam + '_Mom')
        if verbose: print(f'  SO3_T poly+interactions:      {len(new_feats)-n0}')

    if verbose:
        print(f'  New features total:           {len(new_feats)}')
        print(f'  Total features after FE:      {df.shape[1]}')
    return df, new_feats


print('Running feature engineering...')
t0 = time.time()
train_fe, new_feats = feature_engineering(train, lag_map, families, pure_base_cols, verbose=True)
test_fe,  _         = feature_engineering(test,  lag_map, families, pure_base_cols, verbose=False)
print(f'Done in {time.time()-t0:.1f}s')
print(f'Train_FE: {train_fe.shape} | Test_FE: {test_fe.shape}')
print(f'Elapsed: {time.time()-NOTEBOOK_START:.0f}s')

Running feature engineering...
  Momentum (Acc12/Acc23/Div13): 330
  AbsMom (|LagT1|):             110
  Price-specific:               4
  Family aggregates:            20
  SO3_T poly+interactions:      7
  New features total:           471
  Total features after FE:      916
Done in 41.5s
Train_FE: (661574, 916) | Test_FE: (410139, 916)
Elapsed: 75s


In [7]:
# CELL 7 - FINALIZE ARRAYS
train_fe.replace([np.inf, -np.inf], np.nan, inplace=True)
test_fe.replace( [np.inf, -np.inf], np.nan, inplace=True)

assert list(train_fe.columns) == list(test_fe.columns), 'FATAL: Column mismatch!'

feature_cols = train_fe.columns.tolist()
n_features   = len(feature_cols)

print(f'Total features: {n_features}')
print(f'NaN train: {train_fe.isnull().sum().sum():,} | NaN test: {test_fe.isnull().sum().sum():,}')
print('LGB and CatBoost handle NaN natively - no imputation needed')

X      = train_fe.values.astype(np.float32)
X_test = test_fe.values.astype(np.float32)

print(f'X: {X.shape} float32 | X_test: {X_test.shape} float32')
print(f'y: {y.shape} float32')

del train_fe, test_fe, train, test
gc.collect()
print('Memory freed OK')

Total features: 916
NaN train: 0 | NaN test: 0
LGB and CatBoost handle NaN natively - no imputation needed
X: (661574, 916) float32 | X_test: (410139, 916) float32
y: (661574,) float32
Memory freed OK


In [8]:
# CELL 7b - FEATURE SELECTION (OOF IMPORTANCE)
# FIX 6: Reduce 916 -> top features by cumulative gain importance.
#
# WHY: v5 LGB used only ~22 splits across 916 features.
# That means ~890 features had zero importance.
# Keeping them adds noise to tree split decisions and slows training.
#
# METHOD: Train one fast LGB fold (fold 1 split, fixed rounds),
# compute feature gain importance, keep top cumulative 99%.
# Uses TRAINING data only - no test information whatsoever.
#
# POLICY: This is feature selection on training data.
# No test-set cross-sample operations. Fully compliant.

print('Running feature selection (1 fold, fast pass)...')
t_fs = time.time()

# Use fold 1 split for importance estimation (before splitter is defined)
# We define a quick split here; the full CV splitter is defined in Cell 8
if USE_GROUP_CV:
    _splitter_fs = GroupKFold(n_splits=CFG.N_FOLDS)
    _splits_fs   = list(_splitter_fs.split(X, y, groups=cv_groups))
else:
    _kf_fs   = KFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
    _splits_fs = list(_kf_fs.split(X, y))

trn_fs, val_fs = _splits_fs[0]

# Quick LGB pass for importance - fixed 500 rounds, no early stopping
_fs_params = {
    **CFG.LGB_PARAMS,
    'learning_rate':     0.05,  # faster convergence for importance pass
    'min_child_samples': 100,
    'min_gain_to_split': 0.0,
}
_dfs_trn = lgb.Dataset(X[trn_fs], label=y[trn_fs], feature_name=feature_cols, free_raw_data=True)
_dfs_val = lgb.Dataset(X[val_fs],  label=y[val_fs],  reference=_dfs_trn, free_raw_data=True)

_fs_model = lgb.train(
    _fs_params,
    _dfs_trn,
    num_boost_round=500,
    valid_sets=[_dfs_val],
    callbacks=[lgb.log_evaluation(-1)],  # silent
)

# Gain importance: how much each feature contributed to loss reduction
importance_vals = _fs_model.feature_importance(importance_type='gain')
importance_df   = pd.DataFrame({'feature': feature_cols, 'gain': importance_vals})
importance_df   = importance_df.sort_values('gain', ascending=False).reset_index(drop=True)

# Cumulative importance threshold
total_gain      = importance_df['gain'].sum()
importance_df['cum_pct'] = importance_df['gain'].cumsum() / (total_gain + 1e-12)

# Select features: top cumulative 99% OR at least FEAT_MIN_KEEP
mask_thresh  = importance_df['cum_pct'] <= CFG.FEAT_IMPORTANCE_THRESH
n_above_thresh = mask_thresh.sum()
n_keep       = max(CFG.FEAT_MIN_KEEP, n_above_thresh)
selected_features = importance_df['feature'].iloc[:n_keep].tolist()

n_zero = (importance_df['gain'] == 0).sum()
print(f'Features before selection: {len(feature_cols)}')
print(f'Zero-importance features:  {n_zero}')
print(f'Features kept (top {CFG.FEAT_IMPORTANCE_THRESH*100:.0f}% gain): {n_keep}')
print(f'Top 10 features:')
print(importance_df[['feature', 'gain', 'cum_pct']].head(10).to_string(index=False))

# Apply selection to X and X_test
sel_idx    = [feature_cols.index(f) for f in selected_features]
X          = X[:, sel_idx]
X_test     = X_test[:, sel_idx]
feature_cols = selected_features
n_features   = len(feature_cols)

print(f'X after selection: {X.shape}')

# Save importance for inspection
CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
importance_df.to_csv(CFG.OUTPUT_DIR / 'feature_importance.csv', index=False)

del _fs_model, _dfs_trn, _dfs_val, importance_df
gc.collect()
print(f'Feature selection done in {time.time()-t_fs:.1f}s')
print(f'Elapsed: {time.time()-NOTEBOOK_START:.0f}s')

Running feature selection (1 fold, fast pass)...
Features before selection: 916
Zero-importance features:  36
Features kept (top 99% gain): 762
Top 10 features:
                    feature     gain  cum_pct
                S03_A07_A08 3.736854 0.007248
                S02_F03_U01 3.622451 0.014273
S03_D02_A09_A02_B03_E03_E04 3.365017 0.020800
                      Price 3.253994 0.027111
                S01_F03_U01 3.140092 0.033201
S03_D02_A09_A02_B10_E10_E11 2.914279 0.038854
            S03_A03_A02_W02 2.911565 0.044501
S03_D02_A09_A02_B08_E08_E09 2.626342 0.049594
            S03_A07_V01_V09 2.606460 0.054650
S03_D02_A09_A02_B04_E04_E05 2.591756 0.059676
X after selection: (661574, 762)
Feature selection done in 130.8s
Elapsed: 311s


In [9]:
# CELL 7c - Z-SCORE STANDARDIZATION (TRAIN STATS ONLY)
# FIX 7: Standardize features using TRAINING data statistics.
#
# WHY: Features across S01-S05 families have different scales.
# LGB gradient magnitudes are influenced by feature scale.
# Standardization makes gradient signals comparable across families.
#
# HOW: Compute mean/std from FULL TRAINING SET.
# Apply to both X (train) and X_test using TRAIN stats only.
# This is NOT a cross-sample test operation.
# It is equivalent to global mean/std scaling - explicitly ALLOWED
# by competition policy ('global normalization using simple aggregate
# statistics computed over the full test set' is allowed, and
# training-stat normalization is even less controversial).
#
# IMPLEMENTATION: Per-feature. Clip outliers to [-10, 10] after scaling
# to prevent extreme values from dominating gradient.

if CFG.STANDARDIZE:
    print('Applying z-score standardization (train stats only)...')
    t_std = time.time()

    # Compute on TRAINING data only
    train_mean = np.nanmean(X, axis=0).astype(np.float32)
    train_std  = np.nanstd(X,  axis=0).astype(np.float32)
    train_std  = np.where(train_std < 1e-8, 1.0, train_std)  # avoid div/0

    # Apply to train
    X      = ((X      - train_mean) / train_std).astype(np.float32)
    # Apply same stats to test (NO test information used)
    X_test = ((X_test - train_mean) / train_std).astype(np.float32)

    # Clip to [-10, 10]: removes extreme outliers post-standardization
    # without distorting the bulk distribution
    X      = np.clip(X,      -10, 10).astype(np.float32)
    X_test = np.clip(X_test, -10, 10).astype(np.float32)

    # Replace any residual NaN (from constant features) with 0
    X      = np.nan_to_num(X,      nan=0.0)
    X_test = np.nan_to_num(X_test, nan=0.0)

    print(f'  X mean after scaling:  {X.mean():.4f}  (should be ~0)')
    print(f'  X std  after scaling:  {X.std():.4f}  (should be ~1)')
    print(f'  X_test mean:           {X_test.mean():.4f}')
    print(f'  X_test std:            {X_test.std():.4f}')
    print(f'  Done in {time.time()-t_std:.1f}s')
else:
    print('Standardization skipped (CFG.STANDARDIZE=False)')

print(f'Elapsed: {time.time()-NOTEBOOK_START:.0f}s')

Applying z-score standardization (train stats only)...
  X mean after scaling:  -0.0003  (should be ~0)
  X std  after scaling:  0.8999  (should be ~1)
  X_test mean:           -0.0057
  X_test std:            1.0536
  Done in 24.6s
Elapsed: 338s


In [10]:
# CELL 8 - CV SPLITTER
#
# GroupKFold on CV_GROUP:
# - Entire market regimes held out per fold
# - Matches the competition randomized-batch evaluation
# - OOF R2 is honest (no regime leakage into validation)
# - Host confirmed: CV_GROUPs are independent regimes
#
# Why this matters for Final Score = 2 x mean_batch_R2 - full_test_R2:
# CV leakage inflates OOF R2 -> overconfident submission
# GroupKFold gives conservative, reliable OOF estimate

if USE_GROUP_CV:
    splitter = GroupKFold(n_splits=CFG.N_FOLDS)
    splits   = list(splitter.split(X, y, groups=cv_groups))
    print(f'GroupKFold on CV_GROUP ({n_groups} groups -> {CFG.N_FOLDS} folds)')
    for i, (trn_idx, val_idx) in enumerate(splits):
        val_grps = np.unique(cv_groups[val_idx])
        print(f'  Fold {i+1}: {len(trn_idx):,} train | {len(val_idx):,} val | {len(val_grps)} groups')
else:
    kf     = KFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
    splits = list(kf.split(X, y))
    print('KFold fallback (CV_GROUP not found in data)')

print('CV splitter ready OK')

GroupKFold on CV_GROUP (70 groups -> 5 folds)
  Fold 1: 530,676 train | 130,898 val | 14 groups
  Fold 2: 528,902 train | 132,672 val | 14 groups
  Fold 3: 528,902 train | 132,672 val | 14 groups
  Fold 4: 528,908 train | 132,666 val | 14 groups
  Fold 5: 528,908 train | 132,666 val | 14 groups
CV splitter ready OK


In [11]:
# CELL 9 - OVERFIT DIAGNOSTIC (SINGLE FOLD QUICK CHECK)
# Run fold 1 only. Check train vs val gap BEFORE full CV.
# v6 target: gap < 0.01. With fixed params, expect ~200-800 rounds.
# If gap > 0.02 after fixes: further reduce min_child_samples or lambda_l2.
# If best_iter still < 50: min_gain_to_split may need checking in data.

print('OVERFIT DIAGNOSTIC (fold 1, LGB quick check)')
print('-' * 50)

trn_d, val_d = splits[0]

dtrain_d = lgb.Dataset(X[trn_d], label=y[trn_d], feature_name=feature_cols, free_raw_data=True)
dval_d   = lgb.Dataset(X[val_d], label=y[val_d], reference=dtrain_d, free_raw_data=True)

diag_model = lgb.train(
    CFG.LGB_PARAMS,
    dtrain_d,
    num_boost_round=CFG.LGB_NUM_ROUNDS,
    valid_sets=[dtrain_d, dval_d],
    valid_names=['train', 'valid'],
    callbacks=[
        lgb.early_stopping(CFG.LGB_EARLY_STOP, verbose=False),
        lgb.log_evaluation(500),
    ],
)

best_it       = diag_model.best_iteration
train_r2_diag = r2_score(y[trn_d], diag_model.predict(X[trn_d], num_iteration=best_it))
val_r2_diag   = r2_score(y[val_d], diag_model.predict(X[val_d], num_iteration=best_it))
gap           = train_r2_diag - val_r2_diag

print(f'Best iteration: {best_it}')
print(f'Train R2: {train_r2_diag:.6f}')
print(f'Val   R2: {val_r2_diag:.6f}')
print(f'Gap:      {gap:.6f}', end='  ')

if gap < 0.005:   print('EXCELLENT')
elif gap < 0.01:  print('GOOD - acceptable')
elif gap < 0.02:  print('MODERATE - consider more regularization')
else:             print('HIGH - increase lambda_l2 or min_child_samples')

print(f'Elapsed: {time.time()-NOTEBOOK_START:.0f}s / {CFG.RUNTIME_LIMIT_S}s budget')

del diag_model, dtrain_d, dval_d
gc.collect()
print('Proceeding to full 5-fold CV OK')

OVERFIT DIAGNOSTIC (fold 1, LGB quick check)
--------------------------------------------------
Best iteration: 88
Train R2: 0.016498
Val   R2: 0.000359
Gap:      0.016139  MODERATE - consider more regularization
Elapsed: 417s / 1620s budget
Proceeding to full 5-fold CV OK


In [12]:
# CELL 10 - LIGHTGBM 5-FOLD CV
# v6: With fixed params (min_gain_to_split=0.0, min_child_samples=100,
# lambda_l2=3), model should now run 200-800 rounds instead of v5's 22.
# Time estimate: ~50-150s/fold x 5 = ~5-12 min.
# If best_iter < 50 again: the signal may not exist in this feature set.
# Print best_iter per fold to diagnose.

oof_lgb    = np.zeros(len(y), dtype=np.float64)
pred_lgb   = np.zeros(len(X_test), dtype=np.float64)
lgb_scores = []
lgb_iters  = []

print('=' * 50)
print(f'LightGBM - {CFG.N_FOLDS}-Fold CV  (budget: ~10 min)')
print('=' * 50)

for fold, (trn_idx, val_idx) in enumerate(splits):
    elapsed = time.time() - NOTEBOOK_START
    # If more than 35% of budget used before LGB is done, start skipping
    if elapsed > CFG.RUNTIME_LIMIT_S * 0.35 and fold >= 3:
        print(f'Time budget at risk ({elapsed:.0f}s). Stopping LGB after {fold} folds.')
        break

    t0 = time.time()
    dtrain = lgb.Dataset(X[trn_idx], label=y[trn_idx], feature_name=feature_cols, free_raw_data=True)
    dval   = lgb.Dataset(X[val_idx], label=y[val_idx], reference=dtrain, free_raw_data=True)

    model = lgb.train(
        CFG.LGB_PARAMS,
        dtrain,
        num_boost_round=CFG.LGB_NUM_ROUNDS,
        valid_sets=[dtrain, dval],
        valid_names=['train', 'valid'],
        callbacks=[
            lgb.early_stopping(CFG.LGB_EARLY_STOP, verbose=False),
            lgb.log_evaluation(500),
        ],
    )

    best_it          = model.best_iteration
    lgb_iters.append(best_it)
    val_preds        = model.predict(X[val_idx], num_iteration=best_it)
    oof_lgb[val_idx] = val_preds
    pred_lgb        += model.predict(X_test, num_iteration=best_it) / CFG.N_FOLDS

    fold_r2 = r2_score(y[val_idx], val_preds)
    lgb_scores.append(fold_r2)
    print(f'  Fold {fold+1}/{CFG.N_FOLDS} | R2: {fold_r2:+.6f} | iter: {best_it} | {time.time()-t0:.0f}s')

    del model, dtrain, dval
    gc.collect()

n_lgb_folds = len(lgb_scores)
if n_lgb_folds < CFG.N_FOLDS:
    print(f'Only {n_lgb_folds}/{CFG.N_FOLDS} LGB folds completed (time constraint)')
    pred_lgb = pred_lgb * CFG.N_FOLDS / n_lgb_folds
    filled_idx = np.concatenate([splits[i][1] for i in range(n_lgb_folds)])
    lgb_oof_r2 = r2_score(y[filled_idx], oof_lgb[filled_idx])
else:
    lgb_oof_r2 = r2_score(y, oof_lgb)

lgb_std = np.std(lgb_scores)
print(f'LGB OOF R2:    {lgb_oof_r2:+.6f}')
print(f'Fold std:      {lgb_std:.6f}')
print(f'Avg best iter: {np.mean(lgb_iters):.0f}')
print(f'Elapsed:       {time.time()-NOTEBOOK_START:.0f}s')

LightGBM - 5-Fold CV  (budget: ~10 min)
  Fold 1/5 | R2: +0.000359 | iter: 88 | 73s
  Fold 2/5 | R2: +0.001774 | iter: 180 | 86s
  Fold 3/5 | R2: +0.001095 | iter: 87 | 68s
Time budget at risk (687s). Stopping LGB after 3 folds.
Only 3/5 LGB folds completed (time constraint)
LGB OOF R2:    +0.001130
Fold std:      0.000578
Avg best iter: 118
Elapsed:       687s


In [13]:
# CELL 11 - CATBOOST 5-FOLD CV (PRIMARY MODEL)
# Budget: ~23 min for 5 folds
# v4 OOF R2: 0.055276 - far superior to LGB (0.030)
#
# Runtime math:
#   v4: 1154s/fold @ 5000 iter -> 0.231s/iter
#   v5: 1200 iter x 0.231s = 277s/fold x 5 = 23 min
#   With early stopping (od_wait=100): likely 800-1000 actual iter -> ~18 min
#
# Dynamic guard: if time is tight, reduce iterations for remaining folds.
#
# Compliance: predictions are made independently per row.
# No cross-sample information used in scoring test rows.

oof_cat    = np.zeros(len(y), dtype=np.float64)
pred_cat   = np.zeros(len(X_test), dtype=np.float64)
cat_scores = []
cat_iters  = []
CATBOOST_DONE = False

if CATBOOST_AVAILABLE:
    print('=' * 50)
    print(f'CatBoost - {CFG.N_FOLDS}-Fold CV  (PRIMARY, budget: ~23 min)')
    print('=' * 50)

    for fold, (trn_idx, val_idx) in enumerate(splits):
        elapsed        = time.time() - NOTEBOOK_START
        remaining      = CFG.RUNTIME_LIMIT_S - elapsed
        folds_left     = CFG.N_FOLDS - fold
        print(f'  Fold {fold+1} starting | {remaining:.0f}s remaining')

        if remaining < 60:
            print('TIME BUDGET EXHAUSTED - stopping CatBoost early')
            break

        # Dynamically reduce iterations if time budget is tight
        iters_this_fold = CFG.CAT_PARAMS['iterations']
        if folds_left > 0:
            time_per_fold = (remaining - 60) / folds_left  # 60s final buffer
            iters_budget  = int(time_per_fold / CFG.CAT_SECS_PER_ITER)
            iters_this_fold = min(iters_this_fold, max(500, iters_budget))
            if iters_this_fold < CFG.CAT_PARAMS['iterations']:
                print(f'  Reducing to {iters_this_fold} iter (time constraint)')

        cat_params_fold = {**CFG.CAT_PARAMS, 'iterations': iters_this_fold}

        t0 = time.time()
        pool_trn = Pool(X[trn_idx], y[trn_idx])
        pool_val = Pool(X[val_idx], y[val_idx])

        model = CatBoostRegressor(**cat_params_fold)
        model.fit(pool_trn, eval_set=pool_val, use_best_model=True)

        val_preds        = model.predict(X[val_idx])
        oof_cat[val_idx] = val_preds
        pred_cat        += model.predict(X_test) / CFG.N_FOLDS

        fold_r2  = r2_score(y[val_idx], val_preds)
        best_it  = model.get_best_iteration()
        cat_scores.append(fold_r2)
        cat_iters.append(best_it)
        print(f'  Fold {fold+1}/{CFG.N_FOLDS} | R2: {fold_r2:+.6f} | best_iter: {best_it} | {time.time()-t0:.0f}s')

        del model, pool_trn, pool_val
        gc.collect()

        n_cat_folds = len(cat_scores)
        if n_cat_folds == 0:
            print('No CatBoost folds completed — skipping safely')
            cat_oof_r2 = 0.0
            cat_std = 0.0
            pred_cat = np.zeros(len(X_test))
            CATBOOST_DONE = False
        elif n_cat_folds < CFG.N_FOLDS:
            print(f'Only {n_cat_folds}/{CFG.N_FOLDS} folds completed')
            pred_cat = pred_cat * CFG.N_FOLDS / n_cat_folds
            filled = np.concatenate([splits[i][1] for i in range(n_cat_folds)])
            cat_oof_r2 = r2_score(y[filled], oof_cat[filled])
        else:
            cat_oof_r2 = r2_score(y, oof_cat)
        cat_std       = np.std(cat_scores)
        CATBOOST_DONE = True

    print(f'CAT OOF R2:    {cat_oof_r2:+.6f}')
    print(f'Fold std:      {cat_std:.6f}')
    print(f'Avg best iter: {np.mean(cat_iters):.0f}')
    print(f'Elapsed:       {time.time()-NOTEBOOK_START:.0f}s')
else:
    cat_oof_r2 = 0.0
    cat_std    = 0.0
    print('CatBoost not available - LGB-only submission')

CatBoost - 5-Fold CV  (PRIMARY, budget: ~23 min)
  Fold 1 starting | 845s remaining
  Reducing to 679 iter (time constraint)
0:	learn: 0.0296123	test: 0.0290985	best: 0.0290985 (0)	total: 463ms	remaining: 5m 14s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.02909102831
bestIteration = 42

Shrink model to first 43 iterations.
  Fold 1/5 | R2: +0.000447 | best_iter: 42 | 48s
Only 1/5 folds completed
  Fold 2 starting | 797s remaining
  Reducing to 797 iter (time constraint)
0:	learn: 0.0295915	test: 0.0291918	best: 0.0291918 (0)	total: 403ms	remaining: 5m 21s
200:	learn: 0.0293272	test: 0.0291746	best: 0.0291744 (199)	total: 52.8s	remaining: 2m 36s
400:	learn: 0.0291116	test: 0.0291660	best: 0.0291660 (400)	total: 1m 42s	remaining: 1m 41s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.02916367174
bestIteration = 478

Shrink model to first 479 iterations.
  Fold 2/5 | R2: +0.001886 | best_iter: 478 | 162s
Only 2/5 folds completed
  Fold 3 start

In [14]:
# CELL 12 - ENSEMBLE OPTIMIZATION
# OOF-based weight optimization. NO test-set exploitation.
# Weights fitted on training OOF predictions only.
# Softmax ensures positive weights summing to 1.
# 10 random restarts for robustness.

model_names = ['LGB', 'CAT'] if CATBOOST_DONE else ['LGB']
oofs_list   = [oof_lgb, oof_cat] if CATBOOST_DONE else [oof_lgb]
preds_list  = [pred_lgb, pred_cat] if CATBOOST_DONE else [pred_lgb]

print('ENSEMBLE OPTIMIZATION (OOF-based, no test exploitation)')
print('-' * 50)
for name, oof in zip(model_names, oofs_list):
    print(f'  {name}  OOF R2: {r2_score(y, oof):+.6f}')

oof_sa  = np.mean(oofs_list, axis=0)
pred_sa = np.mean(preds_list, axis=0)
r2_sa   = r2_score(y, oof_sa)
print(f'Simple avg OOF R2: {r2_sa:+.6f}')

if len(oofs_list) > 1:
    oofs_arr  = np.stack(oofs_list, axis=1)
    preds_arr = np.stack(preds_list, axis=1)

    def neg_r2(raw_w):
        w     = np.exp(raw_w) / np.exp(raw_w).sum()
        blend = (oofs_arr * w).sum(axis=1)
        return -r2_score(y, blend)

    best_r2_opt, best_w = -np.inf, None
    for seed in range(10):
        rng    = np.random.RandomState(seed)
        w0     = rng.randn(len(oofs_list)) * 0.1
        result = minimize(neg_r2, w0, method='Nelder-Mead',
                          options={'maxiter': 2000, 'xatol': 1e-6, 'fatol': 1e-8})
        if -result.fun > best_r2_opt:
            best_r2_opt = -result.fun
            best_w      = np.exp(result.x) / np.exp(result.x).sum()

    oof_opt  = (oofs_arr  * best_w).sum(axis=1)
    pred_opt = (preds_arr * best_w).sum(axis=1)
    r2_opt   = r2_score(y, oof_opt)

    print(f'Optimized OOF R2:  {r2_opt:+.6f}')
    for name, w in zip(model_names, best_w):
        print(f'  Weight {name}: {w:.4f}')

    if r2_opt > r2_sa:
        final_oof, final_pred = oof_opt, pred_opt
        final_r2, method_str  = r2_opt, 'Optimized wts'
        print(f'Using OPTIMIZED weights (+{r2_opt-r2_sa:.6f})')
    else:
        final_oof, final_pred = oof_sa, pred_sa
        final_r2, method_str  = r2_sa, 'Simple avg'
        print('Using SIMPLE AVERAGE')
else:
    final_oof  = oofs_list[0]
    final_pred = preds_list[0]
    final_r2   = r2_score(y, final_oof)
    method_str = model_names[0] + ' only'
    r2_opt     = final_r2
    print(f'Single model: R2 = {final_r2:+.6f}')

print(f'FINAL OOF R2: {final_r2:+.6f}')
print(f'Elapsed: {time.time()-NOTEBOOK_START:.0f}s')

ENSEMBLE OPTIMIZATION (OOF-based, no test exploitation)
--------------------------------------------------
  LGB  OOF R2: +0.000671
  CAT  OOF R2: +0.001151
Simple avg OOF R2: +0.001220
Optimized OOF R2:  +0.001266
  Weight LGB: 0.3053
  Weight CAT: 0.6947
Using OPTIMIZED weights (+0.000047)
FINAL OOF R2: +0.001266
Elapsed: 1293s


In [15]:
# CELL 13 - SUBMISSION GUARDS AND SAVE
# 6 hard assertions. Any failure = crash before writing.

CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

submission = pd.DataFrame({'ID': test_ids, 'TARGET': final_pred})

assert len(submission)            == len(test_ids),              'FATAL: Wrong row count!'
assert list(submission.columns)  == ['ID', 'TARGET'],           'FATAL: Wrong columns!'
assert submission['ID'].nunique() == len(test_ids),             'FATAL: Duplicate IDs!'
assert not submission['TARGET'].isnull().any(),                  'FATAL: NaN in TARGET!'
assert not np.isinf(submission['TARGET'].values).any(),          'FATAL: Inf in TARGET!'
assert submission['TARGET'].std() > 0,                           'FATAL: Constant predictions!'

print('All 6 guards passed OK')

sub_path = CFG.OUTPUT_DIR / 'submission.csv'
submission.to_csv(sub_path, index=False)

verify = pd.read_csv(sub_path)
assert len(verify) == len(test_ids)
assert list(verify.columns) == ['ID', 'TARGET']

print(f'submission.csv SAVED AND VERIFIED')
print(f'  Path:     {sub_path}')
print(f'  Rows:     {len(submission):,}')
print(f'  ID range: {submission["ID"].min()} to {submission["ID"].max()}')
print()
print('TARGET stats:')
print(submission['TARGET'].describe())
total_elapsed = time.time() - NOTEBOOK_START
print(f'\nTotal elapsed: {total_elapsed:.0f}s = {total_elapsed/60:.1f} min')

All 6 guards passed OK
submission.csv SAVED AND VERIFIED
  Path:     output\submission.csv
  Rows:     410,139
  ID range: 672374 to 1089244

TARGET stats:
count    410139.000000
mean          0.000125
std           0.003668
min          -0.042501
25%          -0.001787
50%           0.000110
75%           0.002015
max           0.043818
Name: TARGET, dtype: float64

Total elapsed: 1303s = 21.7 min


In [16]:
# CELL 14 - FINAL SCORECARD

total_elapsed = time.time() - NOTEBOOK_START

print()
print('=' * 67)
print('  R2 SCORECARD - iRage Short-Horizon Return Prediction v6')
print('=' * 67)
print(f'  Features original:  {445}')
print(f'  Features after FE:  {n_features}')
print(f'  Train rows:         {len(X):,}')
print(f'  Test rows:          {len(X_test):,}')
print(f'  CV strategy:        {"GroupKFold(CV_GROUP)" if USE_GROUP_CV else "KFold(shuffle)"}')
print(f'  Total runtime:      {total_elapsed:.0f}s = {total_elapsed/60:.1f} min')
print('-' * 67)
print('  MODEL OOF R2')
for name, oof in zip(model_names, oofs_list):
    r2  = r2_score(y, oof)
    bar = '#' * min(40, max(0, int(r2 * 800)))
    print(f'  {name:<5} {r2:+.6f}  {bar}')
print('-' * 67)
print('  ENSEMBLE')
if len(oofs_list) > 1:
    print(f'  Simple avg:   {r2_sa:+.6f}')
    print(f'  Optimized:    {r2_opt:+.6f}')
print(f'  FINAL ({method_str}): {final_r2:+.6f}')
print('-' * 67)
print('  OVERFIT CHECK (LGB fold 1)')
print(f'  Train R2: {train_r2_diag:.6f} | Val R2: {val_r2_diag:.6f} | Gap: {gap:.6f}')
print(f'  {"LOW OVERFIT" if gap<0.01 else "CHECK OVERFIT"}')
print('-' * 67)
print('  COMPLIANCE')
print('  OK Data: /kaggle/input/competitions/...')
print('  OK No internet / pip / external data')
print('  OK GroupKFold on CV_GROUP (regime-aware CV)')
print('  OK Zero cross-sample test operations')
print('  OK Row-wise FE only (policy compliant)')
runtime_ok = total_elapsed < 1800
print(f'  {"OK" if runtime_ok else "FAIL"} Runtime {total_elapsed/60:.1f} min {"< 30 min" if runtime_ok else "> 30 min EXCEEDED"}')
print('-' * 67)
print('  Final Score = 2 x mean_batch_R2 - full_test_R2')
print('  Strategy: consistent batch R2, zero test leakage')
print('=' * 67)
print()

if final_r2 > 0.05:   print('INTERPRETATION: EXCEPTIONAL - top leaderboard')
elif final_r2 > 0.03: print('INTERPRETATION: STRONG - prize range')
elif final_r2 > 0.01: print('INTERPRETATION: GOOD - above majority')
elif final_r2 > 0:    print('INTERPRETATION: WEAK - positive signal')
else:                  print('INTERPRETATION: NO SIGNAL - revisit model')

print('\nNOTEBOOK COMPLETE v6')
if np.mean(lgb_iters) < 50:
    print('WARNING: LGB avg best_iter < 50 even after fixes.')
    print('  This indicates true signal is very weak in these features.')
    print('  CatBoost OOF R2 is the reliable score to trust.')


  R2 SCORECARD - iRage Short-Horizon Return Prediction v6
  Features original:  445
  Features after FE:  762
  Train rows:         661,574
  Test rows:          410,139
  CV strategy:        GroupKFold(CV_GROUP)
  Total runtime:      1307s = 21.8 min
-------------------------------------------------------------------
  MODEL OOF R2
  LGB   +0.000671  
  CAT   +0.001151  
-------------------------------------------------------------------
  ENSEMBLE
  Simple avg:   +0.001220
  Optimized:    +0.001266
  FINAL (Optimized wts): +0.001266
-------------------------------------------------------------------
  OVERFIT CHECK (LGB fold 1)
  Train R2: 0.016498 | Val R2: 0.000359 | Gap: 0.016139
  CHECK OVERFIT
-------------------------------------------------------------------
  COMPLIANCE
  OK Data: /kaggle/input/competitions/...
  OK No internet / pip / external data
  OK GroupKFold on CV_GROUP (regime-aware CV)
  OK Zero cross-sample test operations
  OK Row-wise FE only (policy compliant)
 

---
## ONE-SHOT SUBMISSION CHECKLIST (v5)

Before clicking Submit Prediction, verify ALL:

- [ ] Total runtime < 30 min (printed in Cell 14 scorecard)
- [ ] CAT best_iter < 1150 (if hitting 1200 every fold, increase iterations and rerun)
- [ ] LGB fold 1 gap < 0.02 (overfit check Cell 9)
- [ ] Final OOF R2 > 0 (model has positive signal)
- [ ] submission.csv SAVED AND VERIFIED (Cell 13 output)
- [ ] submission.csv has exactly 410,139 rows
- [ ] Two columns only: ID, TARGET
- [ ] No NaN, no Inf in TARGET

---
## Complete Requirement Audit

| Competition Rule | Implementation | Status |
|---|---|---|
| Data at /kaggle/input/competitions/short-horizon-return-prediction-challenge-by-i-rage/ | CFG.DATA_DIR exact path | PASS |
| No internet | Zero external calls | PASS |
| No pip install | Import-only, all standard Kaggle libs | PASS |
| CPU only | task_type=CPU, no GPU params | PASS |
| Run from scratch | No cached artifacts loaded | PASS |
| Generate submission.csv | Cell 13 + 6 guards | PASS |
| Finish in 30 min | Runtime-tuned params + dynamic guard | PASS |
| No cross-sample test ranking | Zero argsort/rank on test | PASS |
| No KNN / clustering on test | Pure row-wise FE | PASS |
| No temporal reconstruction | GroupKFold used, no date sorting | PASS |
| Global test stats allowed | Not even used (all FE is row-wise) | PASS |
| CV_GROUP used | GroupKFold on CV_GROUP | PASS |